# 12 – Recommendations

Esplorazione e data cleaning del dataset `recommendations.csv`.

**Colonne:**
| Colonna | Descrizione |
|---|---|
| `mal_id` | ID dell'anime su MAL (FK → `details.csv`) |
| `recommendation_mal_id` | ID dell'anime raccomandato su MAL (FK → `details.csv`) |

## 1. Import e caricamento dati

In [1]:
import pandas as pd
import numpy as np
from dataset_analyzer import analyze

In [2]:
df_rec = pd.read_csv('../datasets/recommendations.csv')
print(f'Shape: {df_rec.shape}')
df_rec.info()
df_rec.head()

Shape: (105249, 2)
<class 'pandas.DataFrame'>
RangeIndex: 105249 entries, 0 to 105248
Data columns (total 2 columns):
 #   Column                 Non-Null Count   Dtype
---  ------                 --------------   -----
 0   mal_id                 105249 non-null  int64
 1   recommendation_mal_id  105249 non-null  int64
dtypes: int64(2)
memory usage: 1.6 MB


,mal_id,recommendation_mal_id
0,3269,317
1,3269,6922
2,3269,299
3,3269,3446
4,3269,5681


In [3]:
n_originale = len(df_rec)

## 1.1 Rimozione duplicati esatti

Prima dell'analisi per colonna, rimuoviamo le righe con valori identici in **tutte** le colonne, mantenendo solo la prima occorrenza.

In [4]:
mask_dup = df_rec.duplicated(keep=False)
n_righe_coinvolte = mask_dup.sum()
n_gruppi = df_rec[mask_dup].duplicated(keep='first').sum()
n_tenute = n_righe_coinvolte - n_gruppi

print(f'Righe totali coinvolte in duplicazioni : {n_righe_coinvolte:,}')
print(f'  → prime occorrenze mantenute         : {n_tenute:,}')
print(f'  → occorrenze extra rimosse           : {n_gruppi:,}')
print()

df_rec.drop_duplicates(keep='first', inplace=True)
print(f'Righe prima della rimozione : {n_originale:,}')
print(f'Righe dopo la rimozione     : {len(df_rec):,}')

Righe totali coinvolte in duplicazioni : 0
  → prime occorrenze mantenute         : 0
  → occorrenze extra rimosse           : 0

Righe prima della rimozione : 105,249
Righe dopo la rimozione     : 105,249


## 2. Analisi colonna per colonna

### 2.1 `mal_id`

In [5]:
analyze(df_rec['mal_id'])


════════════════════════════════════════════════════════════════════════════════
══════════════════════  🔬  SERIES ANALYSIS  —  'mal_id'  ═══════════════════════
════════════════════════════════════════════════════════════════════════════════

  Series name                    mal_id
  dtype                          int64
  Shape                          105,249
  Index range                    0 … 105248
  Kind                           NUMERIC

 📊 Basic Counts
────────────────────────────────────────────────────────────────────────────────
  Total rows                     105,249
  Non-null values                105,249  [██████████████████████████████] 100.0%
  Null / NaN                     0  (0.00%)
  Zeros                          0  (0.00%)
  Positives                      105,249   (100.00%)
  Negatives                      0   (0.00%)
  Unique values                  9,033  (8.58%)

 📐 Descriptive Statistics
────────────────────────────────────────────────────────────────────

**Nessuna pulizia necessaria.**  
Nessun null, dtype già `int64`. I duplicati sono attesi: uno stesso anime può comparire più volte perché raccomandato insieme a diversi altri anime. Chiave esterna verso `details.csv`.

In [6]:
print(f'Null in mal_id      : {df_rec["mal_id"].isna().sum()}')
print(f'Anime distinti      : {df_rec["mal_id"].nunique():,}')

Null in mal_id      : 0
Anime distinti      : 9,033


### 2.2 `recommendation_mal_id`

In [7]:
analyze(df_rec['recommendation_mal_id'])


════════════════════════════════════════════════════════════════════════════════
═══════════════  🔬  SERIES ANALYSIS  —  'recommendation_mal_id'  ═══════════════
════════════════════════════════════════════════════════════════════════════════

  Series name                    recommendation_mal_id
  dtype                          int64
  Shape                          105,249
  Index range                    0 … 105248
  Kind                           NUMERIC

 📊 Basic Counts
────────────────────────────────────────────────────────────────────────────────
  Total rows                     105,249
  Non-null values                105,249  [██████████████████████████████] 100.0%
  Null / NaN                     0  (0.00%)
  Zeros                          0  (0.00%)
  Positives                      105,249   (100.00%)
  Negatives                      0   (0.00%)
  Unique values                  9,041  (8.59%)

 📐 Descriptive Statistics
─────────────────────────────────────────────────────

**Nessuna pulizia necessaria.**  
Nessun null, dtype già `int64`. I duplicati sono attesi: lo stesso anime può essere raccomandato a partire da diversi anime. Chiave esterna verso `details.csv`.

In [8]:
print(f'Null in recommendation_mal_id : {df_rec["recommendation_mal_id"].isna().sum()}')
print(f'Anime distinti (raccomandati) : {df_rec["recommendation_mal_id"].nunique():,}')

Null in recommendation_mal_id : 0
Anime distinti (raccomandati) : 9,041


### 2.3 Chiave primaria `(mal_id, recommendation_mal_id)`

La coppia `(mal_id, recommendation_mal_id)` identifica univocamente una raccomandazione: lo stesso paio di anime non dovrebbe comparire più di una volta. Verifichiamo dopo la rimozione dei duplicati esatti.

In [9]:
n_pk_dup = df_rec.duplicated(subset=['mal_id', 'recommendation_mal_id'], keep=False).sum()
print(f'Duplicati su (mal_id, recommendation_mal_id): {n_pk_dup}')

if n_pk_dup > 0:
    print('→ Rimozione duplicati sulla chiave primaria...')
    df_rec.drop_duplicates(subset=['mal_id', 'recommendation_mal_id'], keep='first', inplace=True)
    print(f'Righe dopo la rimozione: {len(df_rec):,}')
else:
    print('→ Chiave primaria già univoca, nessuna operazione richiesta.')

Duplicati su (mal_id, recommendation_mal_id): 0
→ Chiave primaria già univoca, nessuna operazione richiesta.


## 3. Data Cleaning

In [10]:
print('=== Riepilogo Dataset Pulito ===')
print(f'Righe originali      : {n_originale:>12,}')
print(f'Righe dopo cleaning  : {len(df_rec):>12,}')
print(f'Righe rimosse totali : {n_originale - len(df_rec):>12,}')
print()
print('Dtypes finali:')
print(df_rec.dtypes)
print()
print('Valori mancanti residui:')
print(df_rec.isnull().sum())

=== Riepilogo Dataset Pulito ===
Righe originali      :      105,249
Righe dopo cleaning  :      105,249
Righe rimosse totali :            0

Dtypes finali:
mal_id                   int64
recommendation_mal_id    int64
dtype: object

Valori mancanti residui:
mal_id                   0
recommendation_mal_id    0
dtype: int64


In [11]:
df_rec.head()

,mal_id,recommendation_mal_id
0,3269,317
1,3269,6922
2,3269,299
3,3269,3446
4,3269,5681


## 4. Salvataggio dataset pulito

In [12]:
df_rec.to_csv('../datasets_cleaned/recommendations_clean.csv', index=False)
print('Salvato: datasets_cleaned/recommendations_clean.csv')

Salvato: datasets_cleaned/recommendations_clean.csv
